# NASA CMAPSS FD004 — Colab evaluation (Neraium)

**Setup:** Upload `test_FD004.txt` and `RUL_FD004.txt` (e.g. to `/content/data/`) or mount Drive, then set `TEST_PATH` and `RUL_PATH` in the next cell.

Plots display inline. Optionally saves PNGs under `OUTPUT_DIR`.

In [ ]:
# --- Editable paths (Colab: use /content/...) ---
TEST_PATH = "/content/data/test_FD004.txt"
RUL_PATH = "/content/data/RUL_FD004.txt"
OUTPUT_DIR = "/content/outputs/fd004"  # set to None to skip saving files
SAVE_PLOTS = True  # if True and OUTPUT_DIR set, save PNGs

import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
COLS = ["unit", "cycle", "op1", "op2", "op3"] + [f"s{i}" for i in range(1, 22)]


def load_fd004(test_path: str, rul_path: str) -> tuple[pd.DataFrame, pd.Series]:
    """Load test matrix and per-unit final RUL (line order = unit 1, 2, ...)."""
    test_df = pd.read_csv(test_path, sep=r"\s+", header=None, engine="python")
    test_df = test_df.dropna(axis=1, how="all")
    n = min(len(COLS), test_df.shape[1])
    test_df = test_df.iloc[:, :n].copy()
    test_df.columns = list(COLS[:n])
    for c in test_df.columns:
        test_df[c] = pd.to_numeric(test_df[c], errors="coerce")

    text = Path(rul_path).read_text(encoding="utf-8")
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    vals = [int(float(ln.split()[0])) for ln in lines]
    rul = pd.Series(vals, index=range(1, len(vals) + 1), name="rul_final")
    return test_df, rul


def compute_true_rul(test_df: pd.DataFrame, rul_df: pd.Series) -> pd.DataFrame:
    """
    rul_df: index = unit id, value = RUL at last observed cycle.
    True RUL at cycle c: RUL(c_max) + (c_max - c).
    """
    out = test_df.copy()
    tr = np.zeros(len(out), dtype=float)
    for uid, g in out.groupby("unit"):
        ix = g.index.values
        c = g["cycle"].values
        c_max = float(np.max(c))
        r_end = float(rul_df.get(int(uid), np.nan))
        if np.isnan(r_end):
            r_end = 0.0
        tr[ix] = r_end + (c_max - c)
    out["true_rul"] = tr
    return out


class _PlaceholderRULModel:
    """Replace with your trained model; must implement predict(X)."""

    def predict(self, X: np.ndarray) -> np.ndarray:
        if X.size == 0:
            return np.array([])
        n = X.shape[0]
        w = float(np.mean(np.abs(X)))
        end_rul = float(np.clip(80.0 - 0.02 * w, 0.0, 125.0))
        return np.linspace(end_rul + (n - 1), end_rul, n)


def get_model():
    return _PlaceholderRULModel()


def preprocess(unit_df: pd.DataFrame) -> np.ndarray:
    """Features for one engine (stub: ops + sensors)."""
    drop = {"unit", "cycle", "true_rul"}
    use = [c for c in unit_df.columns if c not in drop]
    return unit_df[use].to_numpy(dtype=np.float64)


def run_inference(test_df: pd.DataFrame, model) -> tuple[dict[int, float], dict[int, np.ndarray]]:
    """Final predicted RUL per unit + full trajectory per unit."""
    pred_final: dict[int, float] = {}
    pred_traj: dict[int, np.ndarray] = {}
    for uid, g in test_df.groupby("unit"):
        g = g.sort_values("cycle")
        X = preprocess(g)
        traj = np.asarray(model.predict(X), dtype=float)
        pred_traj[int(uid)] = traj
        pred_final[int(uid)] = float(traj[-1]) if len(traj) else float("nan")
    return pred_final, pred_traj

In [ ]:
def _maybe_save(fig, name: str) -> None:
    if SAVE_PLOTS and OUTPUT_DIR:
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        fig.savefig(os.path.join(OUTPUT_DIR, name), dpi=150, bbox_inches="tight")


def plot_pred_vs_true(true_final: np.ndarray, pred_final: np.ndarray) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(true_final))
    ax.plot(x, true_final, "o-", ms=3, label="True RUL (final)")
    ax.plot(x, pred_final, "x-", ms=3, label="Predicted RUL (final)")
    ax.set_xlabel("Engine index (sorted)")
    ax.set_ylabel("RUL")
    ax.set_title("FD004: Predicted vs True RUL (last cycle per engine)")
    ax.legend()
    fig.tight_layout()
    _maybe_save(fig, "fd004_pred_vs_true.png")
    plt.show()


def plot_error_curve(true_final: np.ndarray, pred_final: np.ndarray) -> None:
    err = pred_final - true_final
    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(err))
    ax.plot(x, err, "o-", ms=3, color="C2")
    ax.axhline(0.0, color="gray", lw=0.8)
    ax.set_xlabel("Engine index (sorted)")
    ax.set_ylabel("Predicted − True")
    ax.set_title("FD004: RUL error (final)")
    fig.tight_layout()
    _maybe_save(fig, "fd004_error.png")
    plt.show()


def plot_degradation_curve(
    unit_df: pd.DataFrame,
    unit_preds: np.ndarray,
    unit_id: int = 1,
) -> None:
    g = unit_df[unit_df["unit"] == unit_id].sort_values("cycle")
    cycles = g["cycle"].to_numpy()
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cycles, unit_preds, "-", label=f"Predicted RUL (unit {unit_id})")
    if "true_rul" in g.columns:
        ax.plot(cycles, g["true_rul"].values, "--", alpha=0.8, label="True RUL")
    ax.set_xlabel("Cycle")
    ax.set_ylabel("RUL")
    ax.set_title(f"FD004: Degradation (unit {unit_id})")
    ax.legend()
    fig.tight_layout()
    _maybe_save(fig, f"fd004_degradation_unit{unit_id}.png")
    plt.show()

In [ ]:
# --- Run full workflow (execute after defining paths) ---
assert Path(TEST_PATH).is_file(), f"Missing: {TEST_PATH}"
assert Path(RUL_PATH).is_file(), f"Missing: {RUL_PATH}"

raw_df, rul_series = load_fd004(TEST_PATH, RUL_PATH)
df = compute_true_rul(raw_df, rul_series)
model = get_model()
pred_final_map, pred_traj = run_inference(df, model)

units = sorted(pred_final_map.keys())
true_final_list = []
for u in units:
    sub = df[df["unit"] == u]
    c_max = sub["cycle"].max()
    true_final_list.append(float(sub.loc[sub["cycle"] == c_max, "true_rul"].iloc[0]))
true_final = np.array(true_final_list)
pred_final = np.array([pred_final_map[u] for u in units])

plot_pred_vs_true(true_final, pred_final)
plot_error_curve(true_final, pred_final)
if 1 in pred_traj:
    plot_degradation_curve(df, pred_traj[1], unit_id=1)
else:
    print("Unit 1 not found in test data; skip degradation plot.")

rmse = float(np.sqrt(np.mean((pred_final - true_final) ** 2)))
print(f"RMSE (final RUL): {rmse:.4f}")
if SAVE_PLOTS and OUTPUT_DIR:
    print(f"Plots saved under: {OUTPUT_DIR}")